In [9]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import fbeta_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE

df=pd.read_parquet('train.parquet')
df.columns = df.columns.str.strip().str.lower()
# 1. 최종 선택된 19개 변수 리스트

df['golden_ratio_br'] = df['borrowing dependency'] / (df['retained earnings to total assets'] + 1e-9)

final_vars = ['id',
    'persistent eps in the last four seasons', 'borrowing dependency', 'net income to total assets', 
    'total income/total expense', 'net value growth rate', 'total debt/total net worth', 
    'non-industry income and expenditure/revenue', 'net worth/assets', 'interest expense ratio', 
    'retained earnings to total assets', 'equity to liability', 'after-tax net interest rate', 
    'quick ratio', 'degree of financial leverage (dfl)', "net income to stockholder's equity", 
    'interest coverage ratio (interest expense to ebit)', 'interest-bearing debt interest rate', 
    'inventory/working capital', 'cash/current liability','golden_ratio_br'
]

# 2. 데이터 분할 (X, y 설정)
X = df[final_vars]
y = df['bankrupt?']  # 타겟 컬럼명 확인 필요

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 클래스 불균형 비율 계산 (XGBoost용)
ratio = (y_train == 0).sum() / (y_train == 1).sum()

# 3. Optuna로 찾은 최적 하이퍼파라미터 설정
best_params = {
    'xgb_n_estimators': 712, 'xgb_lr': 0.013410585240898247, 'xgb_max_depth': 7, 
    'xgb_subsample': 0.7774789438634512, 'xgb_colsample': 0.7410703874761337, 
    'lgb_n_estimators': 375, 'lgb_lr': 0.01618709255825155, 'lgb_max_depth': 8, 
    'lgb_subsample': 0.9085133170965529, 'lgb_colsample': 0.9359307444152808, 
    'sampling_strategy': 0.11282883649924065
}

# 4. SMOTE 적용 (최적 비율 적용)
smote = SMOTE(sampling_strategy=best_params['sampling_strategy'], random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# 5. 개별 베이스 모델 정의 및 학습
model_xgb = xgb.XGBClassifier(
    n_estimators=best_params['xgb_n_estimators'],
    learning_rate=best_params['xgb_lr'],
    max_depth=best_params['xgb_max_depth'],
    subsample=best_params['xgb_subsample'],
    colsample_bytree=best_params['xgb_colsample'],
    scale_pos_weight=ratio,
    random_state=42,
    verbosity=0
)

model_lgb = lgb.LGBMClassifier(
    n_estimators=best_params['lgb_n_estimators'],
    learning_rate=best_params['lgb_lr'],
    max_depth=best_params['lgb_max_depth'],
    subsample=best_params['lgb_subsample'],
    colsample_bytree=best_params['lgb_colsample'],
    class_weight='balanced',
    random_state=42,
    verbosity=-1
)

model_xgb.fit(X_train_res, y_train_res)
model_lgb.fit(X_train_res, y_train_res)

# 6. 메타 모델(Stacking) 구성
# 테스트 데이터의 예측 확률을 메타 피처로 사용
meta_test_X = np.column_stack([
    model_xgb.predict_proba(X_test)[:, 1],
    model_lgb.predict_proba(X_test)[:, 1]
])

# 학습 데이터의 예측 확률 (메타 모델 학습용)
meta_train_X = np.column_stack([
    model_xgb.predict_proba(X_train_res)[:, 1],
    model_lgb.predict_proba(X_train_res)[:, 1]
])

meta_model = LogisticRegression(class_weight='balanced', random_state=42)
meta_model.fit(meta_train_X, y_train_res)



final_probs = meta_model.predict_proba(meta_test_X)[:, 1]

# 최적 임계값 설정
target_threshold = 0.03

# 설정된 임계값에 따른 최종 예측값 생성
final_preds = (final_probs >= target_threshold).astype(int)

# 검증셋 결과 확인
current_f2 = fbeta_score(y_test, final_preds, beta=2, zero_division=0)

# 8. 최종 결과 출력
print(f"--- 긴급 복구 결과 ---")
print(f"최종 F2 Score: {current_f2:.4f}") 
print("\n[Confusion Matrix]")
print(confusion_matrix(y_test, final_preds))

--- 긴급 복구 결과 ---
최종 F2 Score: 0.5820

[Confusion Matrix]
[[881  43]
 [  9  22]]


In [15]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import fbeta_score, confusion_matrix

# 1. 교차 검증 설정 (5-Fold)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_scores = []
train_scores = []

print(f"--- 1조 '원픽' 모델 검증 시작 ---")

# 2. OOF 검증 루프
for fold, (train_idx, val_idx) in enumerate(cv.split(X, y), 1):
    # 데이터 분할
    X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
    y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
    
    # SMOTE 적용 (태영님 코드의 최적 비율 0.1128 사용)
    smote = SMOTE(sampling_strategy=0.1128, random_state=42)
    X_res, y_res = smote.fit_resample(X_train_fold, y_train_fold)
    
    # 모델 학습 (XGB + LGBM Stacking 로직 동일 적용)
    model_xgb.fit(X_res, y_res)
    model_lgb.fit(X_res, y_res)
    
    # 메타 모델 학습용 확률 추출
    train_meta = np.column_stack([
        model_xgb.predict_proba(X_res)[:, 1],
        model_lgb.predict_proba(X_res)[:, 1]
    ])
    val_meta = np.column_stack([
        model_xgb.predict_proba(X_val_fold)[:, 1],
        model_lgb.predict_proba(X_val_fold)[:, 1]
    ])
    
    # 메타 모델 학습 및 예측 (임계값 0.03 적용)
    meta_model.fit(train_meta, y_res)
    val_probs = meta_model.predict_proba(val_meta)[:, 1]
    val_preds = (val_probs >= 0.03).astype(int)
    
    # 점수 계산 (F2 Score)
    f2 = fbeta_score(y_val_fold, val_preds, beta=2)
    fold_scores.append(f2)
    
    # Train 점수도 계산 (과적합 확인용)
    train_probs = meta_model.predict_proba(train_meta)[:, 1]
    train_preds = (train_probs >= 0.03).astype(int)
    train_f2 = fbeta_score(y_res, train_preds, beta=2)
    train_scores.append(train_f2)
    
    print(f"Fold {fold}: Train F2 = {train_f2:.4f} | Val F2 = {f2:.4f}")

# 3. 최종 결과 출력
print(f"\n" + "="*30)
print(f"평균 Train F2: {np.mean(train_scores):.4f}")
print(f"평균 CV Score (Val F2): {np.mean(fold_scores):.4f} (±{np.std(fold_scores):.4f})")
print("="*30)

--- 1조 '원픽' 모델 검증 시작 ---
Fold 1: Train F2 = 0.9647 | Val F2 = 0.5464
Fold 2: Train F2 = 0.9647 | Val F2 = 0.5779
Fold 3: Train F2 = 0.9639 | Val F2 = 0.5721
Fold 4: Train F2 = 0.9652 | Val F2 = 0.4749
Fold 5: Train F2 = 0.9612 | Val F2 = 0.6186

평균 Train F2: 0.9639
평균 CV Score (Val F2): 0.5580 (±0.0476)


In [13]:
# ==============================
# 테스트 데이터 예측 및 결과 저장
# ==============================

# 테스트 데이터 로드
test_df = pd.read_parquet('test.parquet')  # 파일명 맞게 수정
test_df.columns = test_df.columns.str.strip().str.lower()

# 파생변수 생성
test_df['golden_ratio_br']= test_df['borrowing dependency'] / (test_df['retained earnings to total assets'] + 1e-9)

# id 저장 후 피처 선택
test_ids = test_df['id']
X_real_test = test_df[final_vars]

# 베이스 모델 예측 확률
meta_real_test_X = np.column_stack([
    model_xgb.predict_proba(X_real_test)[:, 1],
    model_lgb.predict_proba(X_real_test)[:, 1]
])

# 메타 모델로 최종 확률 예측
real_probs = meta_model.predict_proba(meta_real_test_X)[:, 1]

# 학습 시 찾은 최적 임계값 적용
real_preds = (real_probs >= 0.03).astype(int)

# 결과 저장
result_df = pd.DataFrame({
    'ID': test_ids,
    'Bankrupt?': real_preds
})

result_df.to_csv('result11.csv', index=False)
print(f"저장 완료! 예측된 부도 기업 수: {real_preds.sum()}개 / 전체: {len(real_preds)}개")

저장 완료! 예측된 부도 기업 수: 173개 / 전체: 2046개
